# Step 1 condition-alignment evaluation

This notebook evaluates the loss-only audio/text conditioning experiment without changing any checkpoint. It separates three questions:

1. **Condition reliance:** Does the same GT anchor receive higher likelihood with correct rather than corrupted audio/text?
2. **Autoregressive robustness:** Does that advantage survive when the exact same generated motion prefix is supplied to both conditions?
3. **Motion quality:** Does the selected planner improve rollout and oracle-gap anchor-substitution FID without damaging diversity or decoded dynamics?

The primary comparison is the trained P1/P2/P3 checkpoint against a matched q0-q3 P0 CE checkpoint. Do not compare an audio gap obtained with q0-q3 directly against the older q0-only baseline. Anchor FID is an anchor-only diagnostic: non-anchor motion tokens remain GT and Step 2 is not invoked.

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'motion_generation').is_dir() and (candidate / 'SuSuInterActs').is_dir():
            return candidate
    raise FileNotFoundError(f'Could not find sentiAvatar-sandbox above {start}')

PROJECT_ROOT = find_project_root()
OUTPUT_ROOT = PROJECT_ROOT / 'motion_generation/outputs/step1_condition_alignment'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('project:', PROJECT_ROOT)
print('python:', sys.executable)
print('outputs:', OUTPUT_ROOT)

## 1. Configuration

Edit `CHECKPOINTS` first. The candidate below is the default P3 output path. Add the matched P0 checkpoint once it has been trained. Use `best` initially; evaluate `final` or milestone checkpoints only when checkpoint selection is part of the question.

Condition evaluation is enabled by default on all 635 validation clips. Token comparison repeats some forwards, so it is optional. Anchor FID is much more expensive and defaults to a deterministic 128-clip pilot.

In [ ]:
CHECKPOINTS = {
    'candidate_p3_best': PROJECT_ROOT / 'checkpoints/step1_gap3_6k_pretrain_p3_audio_text_cf/best',
    # Recommended matched control after it exists:
    # 'p0_ce_best': PROJECT_ROOT / 'checkpoints/step1_gap3_6k_pretrain_p0_ce/best',
}

DEVICE = 'cuda:0'
SUBSET_SEED = 42
NUM_WORKERS = 0

RUN_CONDITION_EVALUATION = True
CONDITION_MAX_CLIPS = 0       # 0 = complete 635-clip validation set
CONDITION_BATCH_SIZE = 8
SKIP_FIXED_GENERATED_PREFIX = False

RUN_TOKEN_COMPARISON = False # enable after condition evaluation succeeds
TOKEN_TEACHER_MAX_CLIPS = 0
TOKEN_ROLLOUT_MAX_CLIPS = 0
TOKEN_TEACHER_BATCH_SIZE = 32
TOKEN_ROLLOUT_BATCH_SIZE = 8

RUN_ANCHOR_FID = False       # expensive; enable after the earlier gates
FID_MAX_CLIPS = 128          # use 0 only for the final 635-clip report
FID_ROLLOUT_BATCH_SIZE = 8
FID_BATCH_SIZE = 64
METRIC_SEED = 42

assert CHECKPOINTS, 'Add at least one checkpoint'
assert len(CHECKPOINTS) == len(set(CHECKPOINTS)), 'Checkpoint labels must be unique'

In [ ]:
checkpoint_preflight = pd.DataFrame([
    {
        'label': label,
        'path': str(path),
        'exists': path.is_dir(),
        'has_source_config': (path / 'phase1_source_config.json').is_file(),
    }
    for label, path in CHECKPOINTS.items()
])
display(checkpoint_preflight)
missing = checkpoint_preflight.loc[
    ~(checkpoint_preflight['exists'] & checkpoint_preflight['has_source_config']), 'label'
].tolist()
assert not missing, f'Missing or incomplete checkpoints: {missing}'

## 2. Correct-versus-corrupted condition likelihood

For each checkpoint, the evaluator reports

`gap = NLL(GT anchors | corrupted condition) - NLL(GT anchors | correct condition)`

in nats per supervised motion ID. Positive is desired. Audio uses the same clip shifted strictly into its causal past; text uses a duplicate-safe different transcript with identical serialized length. Under `fixed_generated_prefix`, the prefix is generated once with the correct condition and then frozen for both likelihood evaluations.

In [ ]:
condition_dir = OUTPUT_ROOT / 'condition_likelihood'
condition_dir.mkdir(parents=True, exist_ok=True)
condition_jsons = {}
for label, checkpoint in CHECKPOINTS.items():
    output_json = condition_dir / f'{label}.json'
    condition_jsons[label] = output_json
    command = [
        sys.executable,
        str(PROJECT_ROOT / 'motion_generation/scripts/evaluate_step1_condition_alignment.py'),
        '--checkpoint', str(checkpoint),
        '--output_json', str(output_json),
        '--device', DEVICE,
        '--max_clips', str(CONDITION_MAX_CLIPS),
        '--batch_size', str(CONDITION_BATCH_SIZE),
        '--subset_seed', str(SUBSET_SEED),
    ]
    if SKIP_FIXED_GENERATED_PREFIX:
        command.append('--skip_generated_prefix')
    print(' '.join(command))
    if RUN_CONDITION_EVALUATION:
        subprocess.run(command, cwd=PROJECT_ROOT, check=True)
    else:
        print('Skipped; reading an existing JSON if present.')

In [ ]:
condition_rows = []
for label, path in condition_jsons.items():
    if not path.is_file():
        print('No condition result for', label, '-', path)
        continue
    payload = json.loads(path.read_text(encoding='utf-8'))
    for result in payload['results']:
        rollout = result.get('rollout', {})
        gaps = result['condition_gap_nats_per_token']
        counts = result['condition_examples']
        condition_rows.append({
            'checkpoint': label,
            'history': result['history'],
            'tokens': result['tokens'],
            'cross_entropy': result['cross_entropy'],
            'accuracy': result['accuracy'],
            'audio_gap_nats_per_token': gaps.get('audio', np.nan),
            'text_gap_nats_per_token': gaps.get('text', np.nan),
            'audio_examples': counts.get('audio', 0),
            'text_examples': counts.get('text', 0),
            'rollout_accuracy': rollout.get('accuracy', np.nan),
            'rollout_q0_accuracy': rollout.get('q0_accuracy', np.nan),
            'rollout_confidence': rollout.get('mean_confidence', np.nan),
            'rollout_entropy': rollout.get('mean_entropy', np.nan),
        })
condition_results = pd.DataFrame(condition_rows)
assert not condition_results.empty, 'No condition-evaluation JSON was available'
condition_results.to_csv(OUTPUT_ROOT / 'condition_alignment_summary.csv', index=False)
display(condition_results.round(6))

In [ ]:
gap_long = condition_results.melt(
    id_vars=['checkpoint', 'history'],
    value_vars=['audio_gap_nats_per_token', 'text_gap_nats_per_token'],
    var_name='modality',
    value_name='gap_nats_per_token',
)
gap_long['modality'] = gap_long['modality'].str.replace('_gap_nats_per_token', '', regex=False)
gap_pivot = gap_long.pivot(index=['checkpoint', 'history'], columns='modality', values='gap_nats_per_token')
display(gap_pivot.round(6))

ax = gap_pivot.plot(kind='bar', figsize=(12, 5), color=['#4C78A8', '#F58518'])
ax.axhline(0.0, color='black', linewidth=1)
ax.set_ylabel('Correct-condition advantage (nats / motion ID)')
ax.set_title('Condition reliance: positive is desired')
ax.tick_params(axis='x', rotation=25)
plt.tight_layout()
plt.show()

In [ ]:
decision_rows = []
for checkpoint, group in condition_results.groupby('checkpoint'):
    by_history = group.set_index('history')
    teacher = by_history.loc['teacher_forced']
    generated = by_history.loc['fixed_generated_prefix'] if 'fixed_generated_prefix' in by_history.index else None
    row = {
        'checkpoint': checkpoint,
        'teacher_audio_gap': teacher['audio_gap_nats_per_token'],
        'teacher_text_gap': teacher['text_gap_nats_per_token'],
        'teacher_ce': teacher['cross_entropy'],
        'teacher_accuracy': teacher['accuracy'],
    }
    if generated is not None:
        row.update({
            'generated_prefix_audio_gap': generated['audio_gap_nats_per_token'],
            'generated_prefix_text_gap': generated['text_gap_nats_per_token'],
            'generated_prefix_ce': generated['cross_entropy'],
            'rollout_accuracy': generated['rollout_accuracy'],
            'rollout_q0_accuracy': generated['rollout_q0_accuracy'],
            'audio_gap_retention': generated['audio_gap_nats_per_token'] / teacher['audio_gap_nats_per_token'] if teacher['audio_gap_nats_per_token'] > 0 else np.nan,
            'text_gap_retention': generated['text_gap_nats_per_token'] / teacher['text_gap_nats_per_token'] if teacher['text_gap_nats_per_token'] > 0 else np.nan,
        })
    decision_rows.append(row)
condition_decision = pd.DataFrame(decision_rows).set_index('checkpoint')
condition_decision.to_csv(OUTPUT_ROOT / 'condition_alignment_decision_table.csv')
display(condition_decision.round(6))

### Reading the condition table

A useful checkpoint should have positive audio/text gaps under teacher forcing **and** fixed generated prefixes. The generated-prefix value is more important because it tests whether conditioning survives imperfect history. A large teacher gap that collapses or changes sign under generated history is not sufficient. Compare every gap against P0 under the identical evaluator; absolute audio and text magnitudes are not directly comparable because their corruptions and eligible anchors differ.

## 3. Optional standard token/rollout comparison

Enable `RUN_TOKEN_COMPARISON` to reproduce the established teacher-forced and fully generated-rollout tables for exactly the checkpoints configured above. This duplicates rollout computation from the condition evaluator but supplies per-q, per-slot, calibration, persistence, and horizon diagnostics.

In [ ]:
token_dir = OUTPUT_ROOT / 'token_comparison'
token_dir.mkdir(parents=True, exist_ok=True)
token_command = [
    sys.executable,
    str(PROJECT_ROOT / 'motion_generation/scripts/evaluate_step1_multipart_comparison.py'),
    '--output_dir', str(token_dir),
    '--device', DEVICE,
    '--teacher_max_clips', str(TOKEN_TEACHER_MAX_CLIPS),
    '--rollout_max_clips', str(TOKEN_ROLLOUT_MAX_CLIPS),
    '--teacher_batch_size', str(TOKEN_TEACHER_BATCH_SIZE),
    '--rollout_batch_size', str(TOKEN_ROLLOUT_BATCH_SIZE),
    '--num_workers', str(NUM_WORKERS),
    '--subset_seed', str(SUBSET_SEED),
]
for label, checkpoint in CHECKPOINTS.items():
    token_command.extend(['--checkpoint', f'{label}={checkpoint}'])
print(' '.join(token_command))
if RUN_TOKEN_COMPARISON:
    subprocess.run(token_command, cwd=PROJECT_ROOT, check=True)
else:
    print('Token comparison disabled.')

In [ ]:
teacher_csv = token_dir / 'multipart_teacher_forced.csv'
rollout_csv = token_dir / 'multipart_generated_rollout.csv'
if teacher_csv.is_file() and rollout_csv.is_file():
    teacher_tokens = pd.read_csv(teacher_csv)
    rollout_tokens = pd.read_csv(rollout_csv)
    display(teacher_tokens[[
        'checkpoint', 'tokens', 'accuracy', 'top5_accuracy', 'cross_entropy',
        'q0_accuracy', 'q1_accuracy', 'q2_accuracy', 'q3_accuracy',
    ]].round(6))
    display(rollout_tokens[[
        'checkpoint', 'clips', 'accuracy', 'q0_accuracy', 'q1_accuracy',
        'q2_accuracy', 'q3_accuracy', 'teacher_forced_accuracy_same_subset',
        'previous_copy_accuracy_same_subset', 'accuracy_margin_over_previous_copy',
        'accuracy_drop_from_teacher_forcing', 'mean_confidence', 'mean_entropy',
    ]].round(6))
else:
    print('No token-comparison CSVs yet.')

## 4. Optional anchor-substitution FID

Enable only after condition and rollout gates are acceptable. Begin with 128 clips, then set `FID_MAX_CLIPS = 0` for the final 635-clip report. Lower FID is better. The primary reference is `causal_codec_reconstruction`; `seed_hold` is the deployable static baseline, while previous-GT-anchor copy is an oracle diagnostic.

In [ ]:
CODEC_ROOT = PROJECT_ROOT / 'checkpoints/causal_multipart_rvqvae'
CAUSAL_CODECS = {
    part: CODEC_ROOT / f'causal_rvq_{part}_512x4_scratch/model/best.pth'
    for part in ('upper', 'lower', 'feet', 'hands')
}
fid_requirements = {
    **{f'codec:{part}': path for part, path in CAUSAL_CODECS.items()},
    'evaluator_checkpoint': PROJECT_ROOT / 'checkpoints/eval_model/best_model.pt',
    'evaluator_config': PROJECT_ROOT / 'evaluation/config/train_bert_orig.yaml',
    'evaluator_mean': PROJECT_ROOT / 'evaluation/stats/humanml3d/guoh3dfeats/mean.pt',
    'evaluator_std': PROJECT_ROOT / 'evaluation/stats/humanml3d/guoh3dfeats/std.pt',
}
fid_preflight = pd.DataFrame([
    {'item': item, 'path': str(path), 'exists': path.exists()}
    for item, path in fid_requirements.items()
])
display(fid_preflight)
if RUN_ANCHOR_FID:
    missing_fid = fid_preflight.loc[~fid_preflight['exists'], 'item'].tolist()
    assert not missing_fid, f'Missing FID prerequisites: {missing_fid}'

In [ ]:
fid_dir = OUTPUT_ROOT / 'anchor_fid'
fid_command = [
    sys.executable,
    str(PROJECT_ROOT / 'motion_generation/scripts/evaluate_step1_anchor_fid.py'),
    '--output_dir', str(fid_dir),
    '--device', DEVICE,
    '--max_clips', str(FID_MAX_CLIPS),
    '--subset_seed', str(SUBSET_SEED),
    '--rollout_batch_size', str(FID_ROLLOUT_BATCH_SIZE),
    '--fid_batch_size', str(FID_BATCH_SIZE),
    '--metric_seed', str(METRIC_SEED),
    '--num_workers', str(NUM_WORKERS),
]
for label, checkpoint in CHECKPOINTS.items():
    fid_command.extend(['--checkpoint', f'{label}={checkpoint}'])
for part, checkpoint in CAUSAL_CODECS.items():
    fid_command.extend([f'--{part}_ckpt', str(checkpoint)])
print(' '.join(fid_command))
if RUN_ANCHOR_FID:
    subprocess.run(fid_command, cwd=PROJECT_ROOT, check=True)
else:
    print('Anchor FID disabled.')

In [ ]:
fid_csv = fid_dir / 'anchor_fid_metrics.csv'
decoded_csv = fid_dir / 'decoded_metrics_summary.csv'
if fid_csv.is_file():
    fid = pd.read_csv(fid_csv)
    codec_relative = (
        fid[fid['reference'] == 'causal_codec_reconstruction']
        .sort_values('FID_norm_by_GT')
        .reset_index(drop=True)
    )
    display(codec_relative[[
        'condition', 'num_clips', 'FID_norm_by_GT', 'FID_raw',
        'delta_fid_norm_vs_codec_floor', 'Diversity_GT', 'Diversity_Gen',
        'diversity_gap',
    ]].round(6))
    if decoded_csv.is_file():
        decoded = pd.read_csv(decoded_csv)
        display(decoded.round(6))
else:
    print('No anchor-FID results yet.')

## 5. Decision rule

Advance the candidate to a separate self-forcing fine-tune only when:

- its intended generated-prefix audio/text gap improves over matched P0;
- teacher-forced CE and rollout accuracy do not materially regress;
- it does not become worse than previous-anchor copying by a larger margin;
- anchor FID, decoded error, dynamics, and diversity remain acceptable; and
- the result is later confirmed with multiple seeds and downstream Step 2.

Do not choose a checkpoint solely because it has the largest condition gap: a model can enlarge the gap by becoming unusually bad under the corrupted condition without improving correct-condition motion quality.